# Tier 1 Policy Regression Tests
Tests for the 12 new complaint policies added in Spring 2026.
Run from `CAPSTONE2026_SPRING/` root.

In [1]:
import sys
sys.path.insert(0, '.')

from intake_engine.state.templates import make_empty_intake
from intake_engine.policies.policy_selector import get_policy_for_complaint
from intake_engine.policies.complaint_policies import COMPLAINT_POLICIES
from intake_engine.policies.base_complaint_policy import BaseComplaintPolicy
from intake_engine.state.rule_based_parser import build_update_from_answer
from intake_engine.IntakeState import IntakeState

print('Imports OK')
print(f'Total policies loaded: {len(COMPLAINT_POLICIES)}')

Imports OK
Total policies loaded: 19


In [2]:
# ── Helpers ──────────────────────────────────────────────────────────────────

PASS = '\033[92m PASS\033[0m'
FAIL = '\033[91m FAIL\033[0m'

def check(label, condition):
    status = PASS if condition else FAIL
    print(f'{status}  {label}')
    return condition

def make_state(complaint):
    state = IntakeState()
    state.data['chief_complaint_primary'] = complaint
    return state

def apply_answer(state, intent, answer):
    update = build_update_from_answer(intent, answer)
    state.apply_update(update)
    target = intent.replace('ask_', '', 1)
    status = state.data['conversation_meta']['question_status']
    if target not in status:
        status[target] = 'asked_answered'
    return state

def get_policy(complaint):
    return get_policy_for_complaint(complaint)

def first_target(complaint):
    policy = get_policy(complaint)
    state = make_state(complaint)
    return policy.choose_next_target(state.data)

results = []
print('Helpers ready')

Helpers ready


## 1. Policy loading — all 12 new policies present

In [3]:
print('=== Policy Loading ===')
expected = [
    'nausea_vomiting', 'fever', 'syncope', 'palpitations',
    'leg_pain_swelling', 'urinary_symptoms', 'cough', 'rash',
    'altered_mental_status', 'seizure', 'eye_pain', 'ear_pain',
]
for key in expected:
    results.append(check(f'Policy "{key}" in COMPLAINT_POLICIES', key in COMPLAINT_POLICIES))

=== Policy Loading ===
 PASS  Policy "nausea_vomiting" in COMPLAINT_POLICIES
 PASS  Policy "fever" in COMPLAINT_POLICIES
 PASS  Policy "syncope" in COMPLAINT_POLICIES
 PASS  Policy "palpitations" in COMPLAINT_POLICIES
 PASS  Policy "leg_pain_swelling" in COMPLAINT_POLICIES
 PASS  Policy "urinary_symptoms" in COMPLAINT_POLICIES
 PASS  Policy "cough" in COMPLAINT_POLICIES
 PASS  Policy "rash" in COMPLAINT_POLICIES
 PASS  Policy "altered_mental_status" in COMPLAINT_POLICIES
 PASS  Policy "seizure" in COMPLAINT_POLICIES
 PASS  Policy "eye_pain" in COMPLAINT_POLICIES
 PASS  Policy "ear_pain" in COMPLAINT_POLICIES


## 2. Alias routing — patients say things differently

In [4]:
print('=== Alias Routing ===')
alias_cases = [
    ('nausea',                  'nausea_vomiting'),
    ('throwing up',             'nausea_vomiting'),
    ('i keep throwing up',      'nausea_vomiting'),
    ('fever',                   'fever'),
    ('running a fever',         'fever'),
    ('chills',                  'fever'),
    ('i fainted',               'syncope'),
    ('passed out',              'syncope'),
    ('blacked out',             'syncope'),
    ('heart racing',            'palpitations'),
    ('heart is pounding',       'palpitations'),
    ('my leg is swollen',       'leg_pain_swelling'),
    ('leg swelling',            'leg_pain_swelling'),
    ('it burns when i pee',     'urinary_symptoms'),
    ('uti',                     'urinary_symptoms'),
    ('i have a cough',          'cough'),
    ('coughing up blood',       'cough'),
    ('i have a rash',           'rash'),
    ('hives',                   'rash'),
    ('confused',                'altered_mental_status'),
    ('not acting right',        'altered_mental_status'),
    ('i had a seizure',         'seizure'),
    ('convulsion',              'seizure'),
    ('my eye hurts',            'eye_pain'),
    ('blurry vision',           'eye_pain'),
    ('floaters',                'eye_pain'),
    ('ear ache',                'ear_pain'),
    ('ringing in ears',         'ear_pain'),
]
for alias, expected_policy in alias_cases:
    policy = get_policy_for_complaint(alias)
    results.append(check(
        f'"{alias}" -> {expected_policy}',
        policy.policy_name == expected_policy
    ))

=== Alias Routing ===
 PASS  "nausea" -> nausea_vomiting
 PASS  "throwing up" -> nausea_vomiting
 PASS  "i keep throwing up" -> nausea_vomiting
 PASS  "fever" -> fever
 PASS  "running a fever" -> fever
 PASS  "chills" -> fever
 PASS  "i fainted" -> syncope
 PASS  "passed out" -> syncope
 PASS  "blacked out" -> syncope
 PASS  "heart racing" -> palpitations
 PASS  "heart is pounding" -> palpitations
 PASS  "my leg is swollen" -> leg_pain_swelling
 PASS  "leg swelling" -> leg_pain_swelling
 PASS  "it burns when i pee" -> urinary_symptoms
 PASS  "uti" -> urinary_symptoms
 PASS  "i have a cough" -> cough
 PASS  "coughing up blood" -> cough
 PASS  "i have a rash" -> rash
 PASS  "hives" -> rash
 PASS  "confused" -> altered_mental_status
 PASS  "not acting right" -> altered_mental_status
 PASS  "i had a seizure" -> seizure
 PASS  "convulsion" -> seizure
 PASS  "my eye hurts" -> eye_pain
 PASS  "blurry vision" -> eye_pain
 PASS  "floaters" -> eye_pain
 PASS  "ear ache" -> ear_pain
 PASS  "ringi

## 3. Critical followups fire first

In [5]:
print('=== Critical Followups Fire First ===')
critical_first_cases = [
    ('nausea',          'bloody_stool_or_melena'),
    ('fever',           'fever_or_neck_stiffness'),
    ('i fainted',       'chest_pain'),
    ('heart racing',    'chest_pain'),
    ('leg swelling',    'rapid_worsening'),
    ('it burns when i pee', 'fever_or_neck_stiffness'),
    ('i have a cough',  'shortness_of_breath'),
    ('i have a rash',   'fever_or_neck_stiffness'),
    ('confused',        'sudden_severe_onset'),
    ('i had a seizure', 'neurologic_symptoms'),
    ('my eye hurts',    'visual_changes'),
    ('ear ache',        'fever_or_neck_stiffness'),
]
for complaint, expected_target in critical_first_cases:
    target = first_target(complaint)
    results.append(check(
        f'{complaint}: first target = {expected_target}',
        target == expected_target
    ))

=== Critical Followups Fire First ===
 PASS  nausea: first target = bloody_stool_or_melena
 PASS  fever: first target = fever_or_neck_stiffness
 PASS  i fainted: first target = chest_pain
 PASS  heart racing: first target = chest_pain
 PASS  leg swelling: first target = rapid_worsening
 PASS  it burns when i pee: first target = fever_or_neck_stiffness
 PASS  i have a cough: first target = shortness_of_breath
 PASS  i have a rash: first target = fever_or_neck_stiffness
 PASS  confused: first target = sudden_severe_onset
 PASS  i had a seizure: first target = neurologic_symptoms
 PASS  my eye hurts: first target = visual_changes
 PASS  ear ache: first target = fever_or_neck_stiffness


## 4. Wrap-up does NOT fire before critical followups answered

In [6]:
print('=== Wrap-up blocked until critical followups answered ===')
complaints = [
    'nausea', 'fever', 'i fainted', 'heart racing',
    'leg swelling', 'it burns when i pee', 'i have a cough',
    'i have a rash', 'confused', 'i had a seizure',
    'my eye hurts', 'ear ache',
]
for complaint in complaints:
    state = make_state(complaint)
    policy = get_policy(complaint)
    ready = policy.is_ready_to_wrap_up(state.data)
    results.append(check(
        f'{complaint}: wrap-up blocked on empty state',
        not ready
    ))

=== Wrap-up blocked until critical followups answered ===
 PASS  nausea: wrap-up blocked on empty state
 PASS  fever: wrap-up blocked on empty state
 PASS  i fainted: wrap-up blocked on empty state
 PASS  heart racing: wrap-up blocked on empty state
 PASS  leg swelling: wrap-up blocked on empty state
 PASS  it burns when i pee: wrap-up blocked on empty state
 PASS  i have a cough: wrap-up blocked on empty state
 PASS  i have a rash: wrap-up blocked on empty state
 PASS  confused: wrap-up blocked on empty state
 PASS  i had a seizure: wrap-up blocked on empty state
 PASS  my eye hurts: wrap-up blocked on empty state
 PASS  ear ache: wrap-up blocked on empty state


## 5. Yes/No parsing for new critical targets

In [7]:
from intake_engine.policies.target_specs import TARGET_SPECS
print('hemoptysis' in TARGET_SPECS)
print('calf_tenderness_or_warmth' in TARGET_SPECS)
print(f'Total targets: {len(TARGET_SPECS)}')

True
True
Total targets: 83


In [8]:
print('=== Yes/No Parsing for New Targets ===')
yes_no_cases = [
    ('ask_hemoptysis',                              'yes',  True),
    ('ask_hemoptysis',                              'no',   False),
    ('ask_rash_or_petechiae',                       'yes',  True),
    ('ask_rash_or_petechiae',                       'nope', False),
    ('ask_calf_tenderness_or_warmth',               'yes',  True),
    ('ask_calf_tenderness_or_warmth',               'no',   False),
    ('ask_unilateral_leg_swelling',                 'yes',  True),
    ('ask_flank_pain',                              'yes',  True),
    ('ask_flank_pain',                              'no',   False),
    ('ask_hematuria',                               'yes',  True),
    ('ask_urinary_retention',                       'no',   False),
    ('ask_floaters_or_flashes',                     'yes',  True),
    ('ask_recent_eye_trauma',                       'no',   False),
    ('ask_hearing_loss_or_tinnitus',                'yeah', True),
    ('ask_ear_drainage_or_bleeding',                'no',   False),
    ('ask_seizure_history',                         'yes',  True),
    ('ask_antiepileptic_compliance',                'yes',  True),
    ('ask_tongue_or_lip_biting',                    'yes',  True),
    ('ask_incontinence_during_event',               'no',   False),
    ('ask_postictal_confusion',                     'yes',  True),
    ('ask_prodrome_witnessed_loss_of_consciousness','yes',  True),
    ('ask_recent_immobility_or_travel',             'yes',  True),
    ('ask_recent_trauma_or_surgery',                'no',   False),
    ('ask_eye_discharge',                           'yes',  True),
    ('ask_redness',                                 'yes',  True),
    ('ask_headache_with_eye_pain',                  'no',   False),
    ('ask_contact_lens_use',                        'yes',  True),
    ('ask_recent_sleep_deprivation',                'yes',  True),
    ('ask_recent_substance_use',                    'no',   False),
]
from intake_engine.state.rule_based_parser import extract_yes_no_unknown
for intent, answer, expected in yes_no_cases:
    update = build_update_from_answer(intent, answer)
    target = intent.replace('ask_', '', 1)
    path = f'policy_answers.{target}'
    value = update.get('set_fields', {}).get(path)
    results.append(check(
        f'{intent}("{answer}") -> {expected}',
        value == expected
    ))

=== Yes/No Parsing for New Targets ===
 PASS  ask_hemoptysis("yes") -> True
 PASS  ask_hemoptysis("no") -> False
 PASS  ask_rash_or_petechiae("yes") -> True
 PASS  ask_rash_or_petechiae("nope") -> False
 PASS  ask_calf_tenderness_or_warmth("yes") -> True
 PASS  ask_calf_tenderness_or_warmth("no") -> False
 PASS  ask_unilateral_leg_swelling("yes") -> True
 PASS  ask_flank_pain("yes") -> True
 PASS  ask_flank_pain("no") -> False
 PASS  ask_hematuria("yes") -> True
 PASS  ask_urinary_retention("no") -> False
 PASS  ask_floaters_or_flashes("yes") -> True
 PASS  ask_recent_eye_trauma("no") -> False
 PASS  ask_hearing_loss_or_tinnitus("yeah") -> True
 PASS  ask_ear_drainage_or_bleeding("no") -> False
 PASS  ask_seizure_history("yes") -> True
 PASS  ask_antiepileptic_compliance("yes") -> True
 PASS  ask_tongue_or_lip_biting("yes") -> True
 PASS  ask_incontinence_during_event("no") -> False
 PASS  ask_postictal_confusion("yes") -> True
 PASS  ask_prodrome_witnessed_loss_of_consciousness("yes")

## 6. Text field parsing for new targets

In [9]:
print('=== Text Field Parsing for New Targets ===')
text_cases = [
    ('ask_vision_loss_pattern', 'blurry in my left eye only',    'blurry in my left eye only'),
    ('ask_vision_loss_pattern', 'completely gone in both eyes.', 'completely gone in both eyes'),
    ('ask_eye_pain_type',       'sharp stabbing pain.',          'sharp stabbing pain'),
    ('ask_eye_pain_type',       'burning sensation',             'burning sensation'),
    ('ask_unilateral_vs_bilateral', 'just the left leg',         'just the left leg'),
]
for intent, answer, expected in text_cases:
    update = build_update_from_answer(intent, answer)
    target = intent.replace('ask_', '', 1)
    path = f'policy_answers.{target}'
    value = update.get('set_fields', {}).get(path)
    results.append(check(
        f'{intent}("{answer}") -> "{expected}"',
        value == expected
    ))

=== Text Field Parsing for New Targets ===
 PASS  ask_vision_loss_pattern("blurry in my left eye only") -> "blurry in my left eye only"
 PASS  ask_vision_loss_pattern("completely gone in both eyes.") -> "completely gone in both eyes"
 PASS  ask_eye_pain_type("sharp stabbing pain.") -> "sharp stabbing pain"
 PASS  ask_eye_pain_type("burning sensation") -> "burning sensation"
 PASS  ask_unilateral_vs_bilateral("just the left leg") -> "just the left leg"


## 7. Seizure policy — specific red flag progression

In [10]:
print('=== Seizure Policy Progression ===')
state = make_state('i had a seizure')
policy = get_policy('i had a seizure')

results.append(check('Seizure: policy_name is seizure', policy.policy_name == 'seizure'))
results.append(check('Seizure: first target is neurologic_symptoms', policy.choose_next_target(state.data) == 'neurologic_symptoms'))

# Answer all critical followups
for target in policy.critical_followups:
    intent = f'ask_{target}'
    apply_answer(state, intent, 'no')

# Now should move to must_characterize
next_target = policy.choose_next_target(state.data)
results.append(check(
    f'Seizure: after critical followups, moves to must_characterize (got {next_target})',
    next_target in policy.must_characterize
))

# Answer all must_characterize
for target in policy.must_characterize:
    intent = f'ask_{target}'
    apply_answer(state, intent, '2 days ago' if target == 'onset' else 'moderate')

# Now should move to high_priority_followups
next_target = policy.choose_next_target(state.data)
results.append(check(
    f'Seizure: after must_characterize, moves to high_priority (got {next_target})',
    next_target in policy.high_priority_followups
))

=== Seizure Policy Progression ===
 PASS  Seizure: policy_name is seizure
 PASS  Seizure: first target is neurologic_symptoms
 PASS  Seizure: after critical followups, moves to must_characterize (got onset)
 PASS  Seizure: after must_characterize, moves to high_priority (got medications)


## 8. Syncope policy — prodrome target present in critical

In [11]:
print('=== Syncope Policy ===')
policy = get_policy('i fainted')
results.append(check('Syncope: policy_name is syncope', policy.policy_name == 'syncope'))
results.append(check(
    'Syncope: prodrome_witnessed_loss_of_consciousness in critical_followups',
    'prodrome_witnessed_loss_of_consciousness' in policy.critical_followups
))
results.append(check(
    'Syncope: medications NOT in critical_followups',
    'medications' not in policy.critical_followups
))

=== Syncope Policy ===
 PASS  Syncope: policy_name is syncope
 PASS  Syncope: prodrome_witnessed_loss_of_consciousness in critical_followups
 PASS  Syncope: medications NOT in critical_followups


## 9. DVT red flags in leg pain policy

In [12]:
print('=== Leg Pain / Swelling Policy ===')
policy = get_policy('leg swelling')
results.append(check('Leg pain: policy_name is leg_pain_swelling', policy.policy_name == 'leg_pain_swelling'))
results.append(check(
    'Leg pain: calf_tenderness_or_warmth in critical_followups',
    'calf_tenderness_or_warmth' in policy.critical_followups
))
results.append(check(
    'Leg pain: recent_heavy_lifting NOT in critical_followups',
    'recent_heavy_lifting' not in policy.critical_followups
))
results.append(check(
    'Leg pain: recent_immobility_or_travel in high_priority_followups',
    'recent_immobility_or_travel' in policy.high_priority_followups
))

=== Leg Pain / Swelling Policy ===
 PASS  Leg pain: policy_name is leg_pain_swelling
 PASS  Leg pain: calf_tenderness_or_warmth in critical_followups
 PASS  Leg pain: recent_heavy_lifting NOT in critical_followups
 PASS  Leg pain: recent_immobility_or_travel in high_priority_followups


## 10. Rash policy — meningococcemia and anaphylaxis screening

In [13]:
print('=== Rash Policy ===')
policy = get_policy('i have a rash')
results.append(check('Rash: policy_name is rash', policy.policy_name == 'rash'))
results.append(check(
    'Rash: fever_or_neck_stiffness in critical_followups',
    'fever_or_neck_stiffness' in policy.critical_followups
))
results.append(check(
    'Rash: trouble_swallowing in critical_followups',
    'trouble_swallowing' in policy.critical_followups
))
results.append(check(
    'Rash: location in must_characterize',
    'location' in policy.must_characterize
))
results.append(check(
    'Rash: character in must_characterize',
    'character' in policy.must_characterize
))

=== Rash Policy ===
 PASS  Rash: policy_name is rash
 PASS  Rash: fever_or_neck_stiffness in critical_followups
 PASS  Rash: trouble_swallowing in critical_followups
 PASS  Rash: location in must_characterize
 PASS  Rash: character in must_characterize


## 11. Wrap-up fires correctly after all required fields answered

In [14]:
print('=== Wrap-up fires after required fields answered ===')

def simulate_complete_intake(complaint):
    state = make_state(complaint)
    policy = get_policy(complaint)

    all_targets = (
        policy.critical_followups +
        policy.must_characterize +
        policy.high_priority_followups
    )

    for target in all_targets:
        intent = f'ask_{target}'
        if target in ['onset']:
            answer = '2 days ago'
        elif target in ['severity']:
            answer = '6/10'
        elif target in ['duration']:
            answer = '3 days'
        elif target in ['aggravating_factors', 'relieving_factors', 'associated_symptoms']:
            answer = 'nothing specific'
        else:
            answer = 'no'
        apply_answer(state, intent, answer)

    return policy.is_ready_to_wrap_up(state.data)

wrap_up_complaints = [
    'nausea', 'fever', 'i fainted', 'heart racing',
    'leg swelling', 'it burns when i pee', 'i have a cough',
    'i have a rash', 'confused', 'i had a seizure',
    'my eye hurts', 'ear ache',
]
for complaint in wrap_up_complaints:
    ready = simulate_complete_intake(complaint)
    results.append(check(f'{complaint}: wrap-up fires after full intake', ready))

=== Wrap-up fires after required fields answered ===
 PASS  nausea: wrap-up fires after full intake
 PASS  fever: wrap-up fires after full intake
 PASS  i fainted: wrap-up fires after full intake
 PASS  heart racing: wrap-up fires after full intake
 PASS  leg swelling: wrap-up fires after full intake
 PASS  it burns when i pee: wrap-up fires after full intake
 PASS  i have a cough: wrap-up fires after full intake
 PASS  i have a rash: wrap-up fires after full intake
 PASS  confused: wrap-up fires after full intake
 PASS  i had a seizure: wrap-up fires after full intake
 PASS  my eye hurts: wrap-up fires after full intake
 PASS  ear ache: wrap-up fires after full intake


## 12. AMS policy — no circular targets

In [15]:
print('=== Altered Mental Status Policy ===')
policy = get_policy('confused')
results.append(check('AMS: policy_name is altered_mental_status', policy.policy_name == 'altered_mental_status'))
results.append(check(
    'AMS: confusion_or_ams NOT in critical_followups (would be circular)',
    'confusion_or_ams' not in policy.critical_followups
))
results.append(check(
    'AMS: medications NOT in critical_followups',
    'medications' not in policy.critical_followups
))
results.append(check(
    'AMS: sudden_severe_onset in critical_followups',
    'sudden_severe_onset' in policy.critical_followups
))

=== Altered Mental Status Policy ===
 PASS  AMS: policy_name is altered_mental_status
 PASS  AMS: confusion_or_ams NOT in critical_followups (would be circular)
 PASS  AMS: medications NOT in critical_followups
 PASS  AMS: sudden_severe_onset in critical_followups


## 13. Summary

In [16]:
total = len(results)
passed = sum(results)
failed = total - passed
print(f'\n==============================')
print(f'  Results: {passed}/{total} passed')
if failed:
    print(f'  FAILED:  {failed}')
else:
    print(f'  All tests passed!')
print(f'==============================')


  Results: 130/130 passed
  All tests passed!
